# Does professional curiosity take the weekend off?

A walkthrough of the analysis in `src/analyse.py`.

**This notebook narrates; it does not re-implement.** Every number comes from the same
functions the script and the test suite use, so the figures here cannot drift away from
the ones in `reports/`. That is deliberate: a notebook that reimplements its own
analysis is two analyses, and they disagree the moment one is edited.

> The committed data is a **simulated fixture** (this repo was built without outbound
> network access). Run `python src/collect.py` for real Wikimedia data. The statistics
> are real either way; the subject matter only becomes real after that command.


In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path('02-python-pageviews')
sys.path.insert(0, str(PROJECT / 'src'))

from analyse import (load, quality_report, naive_test, paired_weekly_test,
                     autocorrelation, sensitivity)

df, source = load(PROJECT / 'data')
print('real API data' if source['is_real_api_data'] else 'SIMULATED FIXTURE')
df.head()


## 1. Look at the data before testing anything

Counting what is wrong comes first. The first question anyone asks about a chart is
"do you trust the data", and the only good answer is a number.


In [ ]:
q = quality_report(df)
print(f"{q['rows']:,} rows | {q['articles']} articles | {q['date_min']} to {q['date_max']}")
print(f"missing days: {q['total_missing_days']} | spike days: {q['total_spike_days']}")
pd.DataFrame(q['per_article'])[['article','days_present','missing_days','median_views','spike_days']]


## 2. The weekly cycle, by eye

Indexed to each topic class's Monday–Friday mean, so articles of wildly different
sizes are comparable on one axis.


In [ ]:
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
fig, ax = plt.subplots(figsize=(8, 4))
for cls, g in df.groupby('topic_class'):
    prof = g.groupby('day_name')['views'].mean().reindex(order)
    ax.plot(order, 100 * prof / prof[:5].mean(), 'o-', label=cls, lw=2)
ax.axhline(100, ls='--', c='#999', lw=1)
ax.set_ylabel('Views, indexed to Mon-Fri mean = 100')
ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
plt.show()


The gap is obvious. The rest of the notebook is about *how confident we are allowed
to be in it* — which is where the interesting mistake lives.


## 3. The obvious test, and why it is wrong

Pool all weekdays, pool all weekends, run Welch's t-test. It gives an absurdly small
p-value. The absurdity is the clue.


In [ ]:
g = df[df.article == 'Data_analysis']
naive = naive_test(g)
print(f"{naive['test']}")
print(f"  n = {naive['effective_n']:,} days, t = {naive['t']:.1f}, p = {naive['p_value']:.2e}")

acf = autocorrelation(g)
print(f"\nlag-1 autocorrelation (all days)    : {acf['lag1_all_days']:.3f}")
print(f"lag-7 autocorrelation (weekly cycle): {acf['lag7_all_days']:.3f}")
print(f"lag-1 (weekdays only)              : {acf['lag1_weekday_only']:.3f}")


Those days are **not independent observations**. A busy day is followed by another
busy day, because whatever caused it persists. Treating correlated observations as
independent inflates the effective sample size and shrinks the p-value to match.

This is *pseudo-replication*, and it is probably the most common statistical error in
analytics work on time series.


## 4. The paired weekly test

For each ISO week: mean weekday views, mean weekend views, and their ratio. Then test
the log ratio against zero across weeks.

- pairing inside a week removes trend and seasonality **by construction**
- one observation per week respects the autocorrelation
- logs because a ratio is right-skewed and bounded below by zero; the mean of logs
  back-transforms to a geometric mean, the right average for a multiplicative quantity


In [ ]:
paired = paired_weekly_test(g)
print(f"{paired['test']}")
print(f"  n = {paired['n_weeks']} weeks")
print(f"  weekend effect : {paired['weekend_effect_pct']:+.1f}%")
print(f"  95% CI         : [{paired['weekend_effect_pct_ci'][0]:+.1f}%, {paired['weekend_effect_pct_ci'][1]:+.1f}%]")
print(f"  p (t-test)     : {paired['p_value']:.2e}")
print(f"  p (Wilcoxon)   : {paired['p_value_wilcoxon']:.2e}   <- distribution-free cross-check")
print(f"\nnaive p was {naive['p_value']:.2e} -- {abs(np.log10(paired['p_value']) - np.log10(naive['p_value'])):.0f} orders of magnitude apart")


Both reject here, because the effect is enormous. **That is luck, not vindication.**
The reason to care is the case where the true effect is 2%: there the naive test
manufactures significance and the paired test correctly declines to.


## 5. Every article, and the control group

The leisure articles are the control. If they moved the same way, the professional
result would just be "the internet is quieter at weekends".


In [ ]:
rows = []
for article, ga in df.groupby('article'):
    r = paired_weekly_test(ga)
    rows.append({'article': article.replace('_',' '),
                 'class': ga['topic_class'].iloc[0],
                 'weekend_effect_%': round(r['weekend_effect_pct'], 1),
                 'ci_low': round(r['weekend_effect_pct_ci'][0], 1),
                 'ci_high': round(r['weekend_effect_pct_ci'][1], 1),
                 'p': f"{r['p_value']:.1e}"})
pd.DataFrame(rows).sort_values('weekend_effect_%').reset_index(drop=True)


## 6. Sensitivity: does the answer depend on my cleaning choices?

If the headline moves materially under any of these, the headline was an artefact of
a decision I made, not a finding.


In [ ]:
pd.DataFrame(sensitivity(df[df.topic_class == 'professional'])).T


All within 0.8 percentage points. The conclusion is robust to each choice.

## Conclusion

Professional topics lose about **33% of weekday traffic at the weekend**; leisure
topics gain. The pattern holds per article, survives five sensitivity variants, and
agrees between a parametric and a distribution-free test.

**But correlation is not causation.** "People read this at work" is a story. A
competing explanation — weekend browsing skews mobile, and these are long articles —
predicts the same pattern. Distinguishing them needs the device breakdown, which this
API endpoint does not provide. See the README for the full list of limitations.
